In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from selenium.webdriver.support.ui import Select
import time

driver = uc.Chrome(version_main=146)
wait = WebDriverWait(driver, 20)

driver.get("https://vahan.parivahan.gov.in/vahan4dashboard/vahan/vahan/view/reportview.xhtml")

REFRESH = "j_idt74"
STATE_DROPDOWN = "j_idt42_label"
GET_STATES = "#j_idt42_items li"

def open_state_dropdown():
    el = wait.until(EC.element_to_be_clickable((By.ID, STATE_DROPDOWN)))
    driver.execute_script("arguments[0].click();", el)

def get_states():
    return wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, GET_STATES)
    ))

def close_dropdown():
    driver.find_element(By.CLASS_NAME, "ui-grid-row").click()
    time.sleep(0.)

def select_option(dropdown_id, items_id, value):
    # open dropdown
    el = wait.until(EC.element_to_be_clickable((By.ID, f"{dropdown_id}_label")))
    driver.execute_script("arguments[0].click();", el)

    # get options
    items = wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, f"#{items_id} li")
    ))

    for item in items:
        label = item.get_attribute("data-label")
        if label and value.lower() in label.lower():
            driver.execute_script("arguments[0].click();", item)
            time.sleep(1.5)
            close_dropdown()
            return

    raise Exception(f"{value} not found")

def safe_click(locator, retries=3, delay=1):
    for attempt in range(retries):
        try:
            el = wait.until(EC.element_to_be_clickable(locator))
            driver.execute_script("arguments[0].click();", el)
            return
        except StaleElementReferenceException:
            print("Stale element, retrying:", locator)
            time.sleep(delay)
    raise Exception(f"Could not click {locator} after {retries} retries")

def select_month(month_label, retries=3):
    for attempt in range(retries):
        try:
            # Always re-open the dropdown to get a fresh DOM
            safe_click((By.ID, "groupingTable:selectMonth_label"))
            time.sleep(0.4)

            # Re-locate the overlay items each time
            month_items = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#groupingTable\\:selectMonth_items li"))
            )

            # Find the desired month by text
            for m in month_items:
                if m.text.strip() == month_label:
                    driver.execute_script("arguments[0].click();", m)
                    return
        except StaleElementReferenceException:
            print(f"Stale element while selecting {month_label}, retrying...")
            time.sleep(0.5)
    raise Exception(f"Could not select month {month_label} after {retries} retries")

# --- TEST STATE CLICKING ---
open_state_dropdown()
states = get_states()
years = list(range(2025, 2003, -1))  # 2026 → 2005
print("Total states:", len(states))

for i in range(1, len(states)):  # skip "All"
    open_state_dropdown()
    states = get_states()

    state = states[i]
    name = state.get_attribute("data-label")

    if not name:
        continue

    print("Clicking:", name)

    driver.execute_script("arguments[0].click();", state)
    time.sleep(1.5)
    close_dropdown()
   

    for year in years:
        print(f"\n--- YEAR: {year} ---")
        
        # --- Set Y-Axis ---
        safe_click((By.ID, "yaxisVar_label"))
        time.sleep(0.5)
        for opt in driver.find_elements(By.CSS_SELECTOR, "#yaxisVar_items li"):
            if "Maker" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break 
            
        # AFTER selecting Y-axis
        wait.until(EC.presence_of_element_located((By.ID, "xaxisVar_label")))
        time.sleep(1)

        # --- Set X-Axis = Month Wise ---
        safe_click((By.ID, "xaxisVar_label"))   # open dropdown
        time.sleep(0.5)

        for opt in driver.find_elements(By.CSS_SELECTOR, "#xaxisVar_items li"):
            if "Fuel" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break


        # select year type (keep fixed)
        select_option("selectedYearType", "selectedYearType_items", "Calendar")
        time.sleep(0.5)

        # select year dynamically
        select_option("selectedYear", "selectedYear_items", str(year))
        time.sleep(0.5)

        # click refresh
        refresh_btn = wait.until(EC.element_to_be_clickable((By.ID, REFRESH)))
        driver.execute_script("arguments[0].click();", refresh_btn)

        print("Refreshed for year:", year)

        time.sleep(2)
        
        #after that select months
        months = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
        for month in months:
            select_month(month)
            time.sleep(0.9)
            # --- Click Excel download ---
            safe_click((By.ID, "groupingTable:xls"))
            time.sleep(.1)

driver.quit()



Total states: 36
Clicking: Andaman & Nicobar Island(3)

--- YEAR: 2025 ---
Refreshed for year: 2025

--- YEAR: 2024 ---
Refreshed for year: 2024

--- YEAR: 2023 ---
Refreshed for year: 2023

--- YEAR: 2022 ---
Refreshed for year: 2022

--- YEAR: 2021 ---
Refreshed for year: 2021

--- YEAR: 2020 ---
Refreshed for year: 2020

--- YEAR: 2019 ---
Refreshed for year: 2019

--- YEAR: 2018 ---
Refreshed for year: 2018

--- YEAR: 2017 ---
Refreshed for year: 2017

--- YEAR: 2016 ---
Refreshed for year: 2016

--- YEAR: 2015 ---
Refreshed for year: 2015

--- YEAR: 2014 ---
Refreshed for year: 2014

--- YEAR: 2013 ---
Refreshed for year: 2013

--- YEAR: 2012 ---
Refreshed for year: 2012

--- YEAR: 2011 ---
Refreshed for year: 2011

--- YEAR: 2010 ---
Refreshed for year: 2010

--- YEAR: 2009 ---
Refreshed for year: 2009

--- YEAR: 2008 ---
Refreshed for year: 2008

--- YEAR: 2007 ---
Refreshed for year: 2007

--- YEAR: 2006 ---
Refreshed for year: 2006

--- YEAR: 2005 ---
Refreshed for year: 2005


In [ ]:
import os, re, time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

from app.utils import Helper

helper = Helper()

download_dir = r"C:\Users\rando\Downloads" 
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

driver.get("https://vahan.parivahan.gov.in/vahan4dashboard/vahan/vahan/view/reportview.xhtml")

# --- CSS selectors for dropdown items ---
state_items_css  = "#j_idt33_items li"
rto_items_css    = "#selectedRto_items li"
yaxis_items_css  = "#yaxisVar_items li"
xaxis_items_css  = "#xaxisVar_items li"
year_items_css   = "#selectedYear_items li"

# --- Helper for safe clicks ---
def safe_click(locator, retries=3, delay=1):
    for attempt in range(retries):
        try:
            el = wait.until(EC.element_to_be_clickable(locator))
            driver.execute_script("arguments[0].click();", el)
            return
        except StaleElementReferenceException:
            print("Stale element, retrying:", locator)
            time.sleep(delay)
    raise Exception(f"Could not click {locator} after {retries} retries")

# --- Select Month ---
def select_month(month_label, retries=3):
    for attempt in range(retries):
        try:
            # Always re-open the dropdown to get a fresh DOM
            safe_click((By.ID, "groupingTable:selectMonth_label"))
            time.sleep(0.4)

            # Re-locate the overlay items each time
            month_items = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#groupingTable\\:selectMonth_items li"))
            )

            # Find the desired month by text
            for m in month_items:
                if m.text.strip() == month_label:
                    driver.execute_script("arguments[0].click();", m)
                    return
        except StaleElementReferenceException:
            print(f"Stale element while selecting {month_label}, retrying...")
            time.sleep(0.5)
    raise Exception(f"Could not select month {month_label} after {retries} retries")

all_dataframes = []


safe_click((By.ID, "j_idt33_label")) #"j_idt41_label"
time.sleep(0.5)
states = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, state_items_css)))


for i in range(len(states)):
    safe_click((By.ID, "j_idt33_label"))
    time.sleep(0.6)
    states = driver.find_elements(By.CSS_SELECTOR, state_items_css)

    state = states[i]
    print("\nSTATE:", state.text)
    driver.execute_script("arguments[0].click();", state)
    time.sleep(1)

    # --- Wait for RTOs ---
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, rto_items_css)) > 1)
    safe_click((By.ID, "selectedRto_label"))
    time.sleep(1)
    rtos = driver.find_elements(By.CSS_SELECTOR, rto_items_css)

    for r in rtos[:1]:   # skip "All Offices"
        print("RTO:", r.text)
        driver.execute_script("arguments[0].click();", r)
        time.sleep(.5)

        # --- Set Y-Axis ---
        safe_click((By.ID, "yaxisVar_label"))
        time.sleep(0.5)
        for opt in driver.find_elements(By.CSS_SELECTOR, yaxis_items_css):
            if "Maker" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break

        # --- Set X-Axis = Month Wise ---
        safe_click((By.ID, "xaxisVar_label"))   # open dropdown
        time.sleep(0.5)

        for opt in driver.find_elements(By.CSS_SELECTOR, "#xaxisVar_items li"):
            if "Vehicle Category" in opt.text or "Vehicle Category" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break

        # --- Set Year ---
        safe_click((By.ID, "selectedYear_label"))
        time.sleep(1)
        for opt in driver.find_elements(By.CSS_SELECTOR, year_items_css):
            if "2026" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break

        # --- Click Refresh ---
        safe_click((By.ID, "j_idt71"))
        time.sleep(.5)
        print("\tRefreshed for", state.text, "->", r.text)
        
        
        # --- Select Month ---

        months = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
        for month in months:
            select_month(month)
            time.sleep(0.9)
            # --- Click Excel download ---
            safe_click((By.ID, "groupingTable:xls"))
            time.sleep(.1)

driver.quit()